In [ ]:
!pip install llama-index llama-index-vector-stores-qdrant llama-index-embeddings-fastembed llama-index-postprocessor-sentence-transformer-rerank pymupdf
!pip install llama-index-readers-file
!pip install llama-index-embeddings-fastembed llama-index-vector-stores-qdrant fastembed
!pip install llama-index-llms-huggingface
!pip install bitsandbytes accelerate
!pip install llama-index-llms-huggingface llama-index-embeddings-fastembed llama-index-vector-stores-qdrant llama-index-postprocessor-sentence-transformer-rerank accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Step 1 : loading and preprocessing

In [ ]:
import os
import re
import pickle
import logging
from tqdm.notebook import tqdm
from llama_index.core import Document
from llama_index.readers.file import PyMuPDFReader

# --- CONFIGURATION ---
INPUT_DIR = "/content/drive/MyDrive/downloaded_pdfs"
OUTPUT_DIR = "/content/drive/MyDrive/processed_docs" # Where we save checkpoints
ERROR_LOG = "/content/drive/MyDrive/processing_errors.txt"

# Ensure output directory exists
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# --- CLEANING FUNCTIONS ---
def clean_text_content(text):
    """
    Aggressive cleaning for scientific papers.
    """
    if not text:
        return ""

    # 1. Remove References/Bibliography (Truncate rest of text)
    ref_pattern = r'\n\s*(References|Bibliography|LITERATURE CITED)\s*\n'
    match = re.search(ref_pattern, text, re.IGNORECASE)
    if match and match.start() > len(text) * 0.5: # Safety: only truncate if in second half
        text = text[:match.start()]

    # 2. Remove Figure Captions and Table Captions
    text = re.sub(r'(Figure|Fig\.|Table)\s?\d+[:.].*?\n', '', text, flags=re.IGNORECASE)

    # 3. Remove Footprints / DOIs / URLs often found in footers
    text = re.sub(r'doi:10\.\d{4,9}/[-._;()/:A-Z0-9]+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 4. Remove Potential Authors/Affiliations (Heuristic: Email addresses)
    text = re.sub(r'\S+@\S+', '', text)

    # 5. Remove Special Characters / Artifacts (keep basic punctuation)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    # 6. Collapse excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# --- MAIN PROCESSING LOOP ---
def process_documents_safely():
    all_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith('.pdf')]
    print(f"Found {len(all_files)} PDFs in source.")

    processed_files = set(os.listdir(OUTPUT_DIR))
    files_to_process = [f for f in all_files if f"{f}.pkl" not in processed_files]
    print(f"Already processed: {len(processed_files)}")
    print(f"Remaining: {len(files_to_process)}")

    errors = []
    reader = PyMuPDFReader()
    pbar = tqdm(files_to_process, desc="Processing PDFs")

    for filename in pbar:
        file_path = os.path.join(INPUT_DIR, filename)
        save_path = os.path.join(OUTPUT_DIR, f"{filename}.pkl")

        try:
            docs = reader.load_data(file_path=file_path)
            full_text = "\n".join([d.text for d in docs])
            cleaned_text = clean_text_content(full_text)

            if not cleaned_text:
                raise ValueError("Text extraction resulted in empty content")

            final_doc = Document(
                text=cleaned_text,
                metadata={"filename": filename}
            )

            with open(save_path, 'wb') as f:
                pickle.dump(final_doc, f)

        except Exception as e:
            error_msg = f"{filename}: {str(e)}"
            errors.append(error_msg)
            pbar.set_postfix({"Last Error": filename[:10]})

    print("\n--- Processing Complete ---")
    print(f"Successfully processed: {len(files_to_process) - len(errors)} new files")
    print(f"Errors encountered: {len(errors)}")

    if errors:
        print("Saving error log to drive...")
        with open(ERROR_LOG, 'w') as f:
            f.write("\n".join(errors))

process_documents_safely()

# Step 2: Text embedding

In [ ]:
import os
import pickle
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from qdrant_client import QdrantClient
from tqdm.notebook import tqdm

PROCESSED_DIR = "/content/drive/MyDrive/processed_docs"
VECTOR_DB_PATH = "/content/drive/MyDrive/qdrant_db"
CHECKPOINT_FILE = "/content/drive/MyDrive/indexed_files.txt"

embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")
client = QdrantClient(path=VECTOR_DB_PATH)
vector_store = QdrantVectorStore(client=client, collection_name="pubmed_papers")
storage_context = StorageContext.from_defaults(vector_store=vector_store)
node_parser = SentenceSplitter(chunk_size=1024, chunk_overlap=20)

def build_index_robustly(batch_size=50):
    try:
        index = VectorStoreIndex.from_vector_store(
            vector_store=vector_store,
            embed_model=embed_model
        )
        print("Existing index loaded.")
    except:
        print("No index found. Creating new...")
        index = None

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            indexed_files = set(f.read().splitlines())
    else:
        indexed_files = set()

    print(f"Resuming... {len(indexed_files)} files already indexed.")
    all_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.pkl')]
    to_process = [f for f in all_files if f not in indexed_files]

    if not to_process:
        print("All files are already indexed!")
        return index

    current_batch_docs = []
    current_batch_filenames = []

    for filename in tqdm(to_process, desc="Indexing"):
        file_path = os.path.join(PROCESSED_DIR, filename)
        try:
            with open(file_path, 'rb') as f:
                doc = pickle.load(f)
                doc.doc_id = filename
                current_batch_docs.append(doc)
                current_batch_filenames.append(filename)
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            continue

        if len(current_batch_docs) >= batch_size:
            if index is None:
                index = VectorStoreIndex.from_documents(
                    current_batch_docs,
                    storage_context=storage_context,
                    embed_model=embed_model,
                    transformations=[node_parser]
                )
            else:
                nodes = node_parser.get_nodes_from_documents(current_batch_docs)
                index.insert_nodes(nodes)

            with open(CHECKPOINT_FILE, 'a') as f:
                for fname in current_batch_filenames:
                    f.write(f"{fname}\n")

            current_batch_docs = []
            current_batch_filenames = []

    if current_batch_docs:
        if index is None:
            index = VectorStoreIndex.from_documents(current_batch_docs, storage_context=storage_context, embed_model=embed_model, transformations=[node_parser])
        else:
            nodes = node_parser.get_nodes_from_documents(current_batch_docs)
            index.insert_nodes(nodes)

        with open(CHECKPOINT_FILE, 'a') as f:
            for fname in current_batch_filenames:
                f.write(f"{fname}\n")

    print("Indexing Complete. Progress saved.")
    return index

index = build_index_robustly()

#Step 3 : Smart retrieval system

In [ ]:
import torch
import os
from huggingface_hub import login
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import VectorStoreIndex, Settings
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from qdrant_client import QdrantClient

VECTOR_DB_PATH = "/content/drive/MyDrive/qdrant_db"

print("Please login to Hugging Face...")
login()

model_name = "meta-llama/Llama-3.2-3B-Instruct"

llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    context_window=8192,
    max_new_tokens=512,
    model_kwargs={
        "torch_dtype": torch.float16,
        "load_in_8bit": False,
        "load_in_4bit": False,
    },
    generate_kwargs={
        "temperature": 0.2,
        "do_sample": True,
        "repetition_penalty": 1.15
    },
    system_prompt = (
    "You are a skilled scientific research assistant. "
    "Your goal is to synthesize the provided context into a clear, helpful explanation for the user. "
    "Rules:\n"
    "1. Do not start with 'According to the provided context'. Jump straight into the answer.\n"
    "2. Group related concepts together.\n"
    "3. Explain the mechanisms simply.\n"
    "4. Prioritize newer science while acknowledging foundational concepts.\n"
    "5. Cite the filenames in parentheses after key statements."
),
    device_map="auto",
)

Settings.llm = llm
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")

if os.path.exists(VECTOR_DB_PATH):
    client = QdrantClient(path=VECTOR_DB_PATH)
    vector_store = QdrantVectorStore(client=client, collection_name="pubmed_papers")
    index = VectorStoreIndex.from_vector_store(
        vector_store=vector_store,
        embed_model=Settings.embed_model
    )
    print("Database loaded successfully.")
else:
    raise FileNotFoundError(f"Could not find database at {VECTOR_DB_PATH}.")

retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=25,
)

# --- OFFLINE LOCAL RERANKER (Replaced Cohere) ---
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=5
)

query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    node_postprocessors=[reranker],
    llm=llm
)

print("\n✅ System Ready! Fully offline local RAG active.")

# Small Models

## Evaluating Llama-3.2-3B:

In [ ]:
import torch
import logging
import os
from google.colab import userdata
from huggingface_hub import login
from transformers import BitsAndBytesConfig
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import VectorStoreIndex, Settings
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from qdrant_client import QdrantClient
from IPython.display import display, Markdown

VECTOR_DB_PATH = "/content/drive/MyDrive/qdrant_db"
login()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

user_oriented_prompt = (
    "You are a highly literal scientific data extraction assistant. "
    "CRITICAL INSTRUCTION: You must base your answer strictly on modern studies, prioritizing those from 2021 to 2024. "
    "If the retrieved context contains outdated mechanisms, IGNORE THEM completely. "
    "1. Do not start with 'According to the provided context'. Jump straight into the answer.\n"
    "2. Under NO circumstances should you use your own general knowledge. If the answer is not in the text, your ENTIRE response must simply be: 'I cannot find the answer to this in the provided documents.'"
)

llm = HuggingFaceLLM(
    model_name="meta-llama/Llama-3.2-3B-Instruct",
    tokenizer_name="meta-llama/Llama-3.2-3B-Instruct",
    context_window=8192,
    max_new_tokens=512,
    model_kwargs={"quantization_config": bnb_config},
    generate_kwargs={
        "temperature": 0.2,
        "do_sample": True,
        "repetition_penalty": 1.15},
    system_prompt=user_oriented_prompt,
    device_map="auto",
)

Settings.llm = llm
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")
logging.getLogger("transformers").setLevel(logging.ERROR)

client = QdrantClient(path=VECTOR_DB_PATH)
vector_store = QdrantVectorStore(client=client, collection_name="pubmed_papers")
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)

retriever = VectorIndexRetriever(index=index, similarity_top_k=25)
reranker = SentenceTransformerRerank(model="cross-encoder/ms-marco-MiniLM-L-6-v2", top_n=5)

query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    node_postprocessors=[reranker],
    llm=llm
)

def ask_malaria_bot(question):
    print(f"🔍 Analyzing documents for: '{question}'...\n")
    try:
        response = query_engine.query(question)
        display(Markdown(f"### 🧬 Answer:\n{response.response}"))
    except Exception as e:
        print(f"Error: {e}")

# Qwen-2.5-3B-Instruct

In [ ]:
import torch
import logging
import os
from google.colab import userdata
from huggingface_hub import login
from transformers import BitsAndBytesConfig
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import VectorStoreIndex, Settings
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.fastembed import FastEmbedEmbedding
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from qdrant_client import QdrantClient
from IPython.display import display, Markdown

VECTOR_DB_PATH = "/content/drive/MyDrive/qdrant_db"
login()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

user_oriented_prompt = (
    "You are a highly literal scientific data extraction assistant. "
    "Rules:\n"
    "1. Begin your answer immediately with the core biological facts. Omit introductory phrases.\n"
    "2. Synthesize only what is explicitly written in the documents.\n"
    "3. If the text does not explicitly contain the answer, you MUST output this exact string and nothing else: 'I cannot find the answer to this in the provided documents.'"
)

def qwen_messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        prompt += f"<|im_start|>{message.role}\n{message.content}<|im_end|>\n"
    if not prompt.endswith("<|im_start|>assistant\n"):
        prompt += "<|im_start|>assistant\n"
    return prompt

def qwen_completion_to_prompt(completion):
    return f"<|im_start|>system\n{user_oriented_prompt}<|im_end|>\n<|im_start|>user\n{completion}<|im_end|>\n<|im_start|>assistant\n"

model_name = "Qwen/Qwen2.5-3B-Instruct"

llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    context_window=4096,
    max_new_tokens=512,
    model_kwargs={
        "quantization_config": bnb_config,
        "attn_implementation": "eager"
    },
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        "repetition_penalty": 1.05
    },
    messages_to_prompt=qwen_messages_to_prompt,
    completion_to_prompt=qwen_completion_to_prompt,
    device_map="auto",
)

Settings.llm = llm
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")
logging.getLogger("transformers").setLevel(logging.ERROR)

client = QdrantClient(path=VECTOR_DB_PATH)
vector_store = QdrantVectorStore(client=client, collection_name="pubmed_papers")
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)

retriever = VectorIndexRetriever(index=index, similarity_top_k=25)
reranker = SentenceTransformerRerank(model="cross-encoder/ms-marco-MiniLM-L-6-v2", top_n=5)

query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    node_postprocessors=[reranker],
    llm=llm
)

def ask_malaria_bot(question):
    print(f"🔍 Analyzing documents for: '{question}'...\n")
    try:
        response = query_engine.query(question)
        display(Markdown(f"### 🧬 Answer:\n{response.response}"))
    except Exception as e:
        print(f"Error: {e}")